# Audio-Aware Subtitle Correction — Colab quick start
Runs the heavy parts (ASR caching, evaluation, training our own models, demo with a public link) on a free T4.
1. Zip the repo (without `.venv`, `wheels`, `cache`) and upload it to Google Drive as `audio-aware-subtitles.zip`.
2. Runtime -> Change runtime type -> T4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/audio-aware-subtitles.zip -d /content
%cd /content/audio-aware-subtitles

## Install (Colab already ships CUDA torch)

In [ ]:
!pip -q install -r requirements.txt
!python -m spacy download en_core_web_sm -q   # ERRANT (evaluation only)
!pip -q install --no-deps -e .
!python -c "import torch, ctranslate2; print(torch.cuda.is_available(), ctranslate2.get_cuda_device_count())"

## Sanity check
Unit tests, then one clip in Mode B (audio only) and Mode A (audio + subtitles).

In [ ]:
!python -m pytest -q -m 'not integration'

In [ ]:
!python -m subfix.cli correct --media tests/fixtures/clip1.flac --out outputs/demo
!python -m subfix.cli correct --media tests/fixtures/clip1.flac --subs tests/fixtures/clip1.corrupted.srt --out outputs/demo

## Benchmark + evaluation
MFA references are easiest to build locally (conda). Copy `data/refs` from your machine, or use `--ref-aligner ctc` here (our CTC aligner; flagged as biased in the results).

In [ ]:
!python tools/build_benchmark.py --stages fetch clips asr
!python tools/build_benchmark.py --stages refs corrupt manifest --ref-aligner ctc
!python experiments/tune_fusion.py --stages features grid
!python experiments/run_eval.py --manifest test --systems noop b3_raw_whisper ours

## Train our own models
Everything except Whisper is trained from scratch (see README). The text models need only the
LibriSpeech-PC manifests; the CTC aligner needs LibriSpeech train-clean-100 (6.3 GB). All runs resume
from `models/*/last.pt`; copy `models/` back to Drive.

In [ ]:
!python experiments/train_lm.py
!python experiments/train_pos.py
!python experiments/train_punct.py
!python experiments/train_corrector.py
!python tools/fetch_data.py --libri train-clean-100
!python experiments/train_aligner.py
!cp -r models /content/drive/MyDrive/subfix-models

## Web app
Builds the React frontend (Node.js is preinstalled on Colab), starts the backend and opens it through Colab's port proxy.

In [ ]:
!cd frontend && npm install --no-audit --no-fund && npm run build
import subprocess, sys
server = subprocess.Popen([sys.executable, '-m', 'app.backend', '--no-browser', '--port', '8000'])
from google.colab import output
output.serve_kernel_port_as_window(8000)